<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/colab/03_run_meeting_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Run meeting LLM (Gemma 4 hackathon)

> **Note:** Prefer [`02_run_meeting_llm.ipynb`](02_run_meeting_llm.ipynb) for new bookmarks. This file keeps the old **03** name so older Colab links still open.

**Goal:** One clean pass from **meeting audio** plus **PDFs** (and optional images) into structured output. Gemma reads the **audio and PDFs directly**—no separate transcript step for now.

**How it works:** Build a message with your policy prompt (`prompts/policy_analysis_v1.md`), optional **Orbis** lookup text, then attach files in order (audio first, then PDFs and images). Call Google’s **Gen AI** API, split the reply on `---DOCUMENT_BREAK---`, and save JSON and short summaries under `03_processed_outputs/`.

**Where to put files:** `03_processed_outputs/01_meeting_media/` on your pipeline Drive (or folder), or set `GOVERNANCE_MEETING_MEDIA_DIR` to any folder path.

**API key:** Colab **Secrets** or your shell—`GEMINI_API_KEY` or `GOOGLE_API_KEY`. Pick the exact **Gemma 4** model name from [Google AI Studio](https://aistudio.google.com/) if the default id does not match your project.

## 1) Find the repo and (on Colab) mount Drive

Same idea as `01_init_drive_layout.ipynb` using `colab_paths.py`. On your own machine, just run from the repo; Colab will try to mount Google Drive when needed.

In [ ]:
# Find open-navigator (Colab clones to /content/open-navigator if needed).
import importlib.util
import os
import sys
from pathlib import Path


def _import_colab_bootstrap():
    candidates: list[Path] = []
    colab_boot = Path("/content/open-navigator/scripts/colab/colab_bootstrap.py")
    if colab_boot.is_file():
        candidates.append(colab_boot)
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        p = base / "scripts" / "colab" / "colab_bootstrap.py"
        if p.is_file():
            candidates.append(p)
            break
    if not candidates:
        try:
            import google.colab  # noqa: F401

            dest = Path("/content/open-navigator")
            marker = dest / "scripts" / "colab" / "colab_bootstrap.py"
            if not marker.is_file():
                print(f"Cloning open-navigator into {dest}…")
                rc = os.system(
                    "git clone https://github.com/getcommunityone/open-navigator.git "
                    f"{dest}"
                )
                if rc != 0:
                    raise RuntimeError(f"git clone failed (exit {rc})")
            if marker.is_file():
                candidates.append(marker)
        except ImportError:
            pass
    if not candidates:
        raise RuntimeError(
            "Could not find scripts/colab/colab_bootstrap.py.\n"
            "Colab: use 02_run_meeting_llm.ipynb or re-run this cell.\n"
            "Local/Cursor: os.environ['OPEN_NAVIGATOR_ROOT'] = '/path/to/open-navigator'"
        )
    mod_path = candidates[0]
    spec = importlib.util.spec_from_file_location("colab_bootstrap", mod_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load {mod_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


_cb = _import_colab_bootstrap()
REPO_PATH = _cb.bootstrap_repo()

from scripts.colab.colab_paths import maybe_mount_google_drive, setup_notebook_paths

maybe_mount_google_drive()
PATHS = setup_notebook_paths()
print("Runtime:", "Google Colab" if PATHS.in_colab else "Local")
print("Repo:                  ", PATHS.project_path)
print("Governance data root: ", PATHS.governance_pipeline_data)


In [ ]:
import os
import sys

# Flip to False if you do not want a hard reset to origin/main.
RUN_GIT_UPDATE = True

proj = PATHS.project_path
if RUN_GIT_UPDATE and proj.is_dir() and (proj / ".git").is_dir():
    print("🔄 Fetching latest code from GitHub...")
    !cd {proj} && git config core.hooksPath /dev/null && git fetch origin && git reset --hard origin/main
elif RUN_GIT_UPDATE:
    print("⚠️ Skip git: not a git checkout at", proj)

assert (proj / "prompts" / "policy_analysis_v1.md").is_file(), (
    f"Missing {proj}/prompts/policy_analysis_v1.md"
)

sys.path.insert(0, str(proj))
COLAB_DIR = proj / "scripts" / "colab"
sys.path.insert(0, str(COLAB_DIR))

os.environ["GOVERNANCE_PIPELINE_DATA_ROOT"] = str(PATHS.governance_pipeline_data)

from scripts.utils.gdrive_paths import GovernancePipelinePaths

PIPE = GovernancePipelinePaths.resolve()
PIPE.ensure_dirs()

from governance_meeting_llm import (
    call_google_genai_multimodal,
    load_text_file,
    merge_orbis_into_organizations,
    parse_policy_analysis_response,
)

PROMPT_PATH = proj / "prompts" / "policy_analysis_v1.md"
POLICY_PROMPT = load_text_file(PROMPT_PATH)
print("Loaded prompt chars:", len(POLICY_PROMPT))
print("Pipeline data root:", PIPE.root)

## 2) Install the Google Gen AI library

This is the small client Google ships for AI Studio–style calls (text plus file attachments).

In [ ]:
%pip install -q "google-genai>=1.0"

## 3) API key

Colab: add a secret named `GEMINI_API_KEY` from [AI Studio](https://aistudio.google.com/apikey).  
Local: export `GEMINI_API_KEY` or `GOOGLE_API_KEY` before you start Jupyter.

In [ ]:
# Optional: point at a different folder or model without using Colab Secrets.
# os.environ["GOVERNANCE_MEETING_MEDIA_DIR"] = "/content/drive/MyDrive/.../01_meeting_media"
# os.environ["GOVERNANCE_GENAI_MODEL"] = "gemma-4-26b-a4b-it"

In [ ]:
import os

try:
    from google.colab import userdata

    _get_secret = userdata.get
except ImportError:

    def _get_secret(name: str):
        return os.environ.get(name)


GEMINI_API_KEY = (
    _get_secret("GEMINI_API_KEY")
    or _get_secret("GOOGLE_API_KEY")
    or os.environ.get("GEMINI_API_KEY")
    or os.environ.get("GOOGLE_API_KEY")
)
if not GEMINI_API_KEY:
    raise RuntimeError(
        "Set GEMINI_API_KEY (Colab Secret or environment). "
        "Create a key at https://aistudio.google.com/apikey"
    )

# Hackathon default: swap for whatever Gemma 4 id your AI Studio project lists.
GENAI_MODEL = os.environ.get("GOVERNANCE_GENAI_MODEL", "gemma-4-26b-a4b-it").strip()
print("GenAI model:", GENAI_MODEL)

## 4) Put your meeting files in one folder

**Audio:** add one main recording (mp3, wav, m4a, …). If there are several, we take the first when sorted by file name—rename if you need a different one first.

**PDFs and pictures:** add any agendas or slides you want the model to read **as real PDFs and images**, not turned into something else.

**Orbis (optional):** if you have `02_reference_data/orbis_files/orbis_lookup_by_org_id.json`, we paste a trimmed copy into the text part of the prompt so org ids can line up with your registry.

**Where:** default folder is `03_processed_outputs/01_meeting_media/` under your pipeline root. Or set `GOVERNANCE_MEETING_MEDIA_DIR` to any full path.

**Outputs:** JSON and raw model text go to `03_processed_outputs/02_gemma_json/`; short markdown summaries go to `03_processed_outputs/03_human_summaries/`.

In [ ]:
# List audio / PDF / image files and build the list of attachments for the API call.
from pathlib import Path
import json
import mimetypes
import os

OUT_GEMMA_JSON = PIPE.gemma_json
OUT_SUMMARIES = PIPE.human_summaries
ORBIS_PATH = PIPE.orbis_files / "orbis_lookup_by_org_id.json"

MEETING_MEDIA = Path(
    os.environ.get("GOVERNANCE_MEETING_MEDIA_DIR", "").strip()
    or (PIPE.root / "03_processed_outputs" / "01_meeting_media")
).expanduser()
MEETING_MEDIA.mkdir(parents=True, exist_ok=True)

AUDIO_EXTS = {".mp3", ".wav", ".m4a", ".aac", ".flac", ".ogg", ".webm"}
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif", ".bmp"}


def _orbis_prompt_blob() -> str | None:
    if not ORBIS_PATH.is_file():
        return None
    data = json.loads(ORBIS_PATH.read_text(encoding="utf-8"))
    s = json.dumps(data, ensure_ascii=False)
    cap = 12_000
    if len(s) <= cap:
        return s
    return s[:cap] + "\n...[orbis JSON truncated for context window]"


def _mime_for(p: Path) -> str:
    guessed, _ = mimetypes.guess_type(p.name)
    if guessed:
        return guessed
    ext = p.suffix.lower()
    if ext == ".pdf":
        return "application/pdf"
    if ext in IMAGE_EXTS:
        return "image/jpeg" if ext in (".jpg", ".jpeg") else "image/png"
    if ext in AUDIO_EXTS:
        return "audio/mpeg" if ext == ".mp3" else "audio/wav"
    return "application/octet-stream"


def _gather_media() -> tuple[list[tuple[Path, str]], Path]:
    if not MEETING_MEDIA.is_dir():
        raise FileNotFoundError(f"Meeting media folder missing: {MEETING_MEDIA}")
    files = sorted([p for p in MEETING_MEDIA.iterdir() if p.is_file()])
    if not files:
        raise FileNotFoundError(
            f"No files in {MEETING_MEDIA}. Add meeting audio + optional PDFs/images."
        )
    audio = sorted([p for p in files if p.suffix.lower() in AUDIO_EXTS])
    if not audio:
        raise FileNotFoundError(
            f"No audio ({', '.join(sorted(AUDIO_EXTS))}) in {MEETING_MEDIA}."
        )
    primary = audio[0]
    media: list[tuple[Path, str]] = [(primary, _mime_for(primary))]
    for p in files:
        ext = p.suffix.lower()
        if p == primary:
            continue
        if ext == ".pdf" or ext in IMAGE_EXTS:
            media.append((p, _mime_for(p)))
    return media, primary


media_paths, primary_audio = _gather_media()
orbis_blob = _orbis_prompt_blob()

print("Meeting media dir:", MEETING_MEDIA)
print("Primary audio:", primary_audio.name)
print("Also sending:", [p.name for p, _ in media_paths[1:]])
print("Orbis JSON pasted into prompt:", "yes" if orbis_blob else "no")
print("JSON out:", OUT_GEMMA_JSON)
print("Summaries:", OUT_SUMMARIES)

## 5) Call Gemma

We send: long policy instructions and Orbis snippet as **text**, then the **audio file**, then each **PDF or image** as its own attachment. Gemma answers in the format your prompt asks for (here: JSON block, then `---DOCUMENT_BREAK---`, then summary pieces).

In [ ]:
from datetime import datetime, timezone

# Short system line: who the model should “be” while it follows your long prompt file.
SYSTEM = (
    "You are an expert political scientist and data architect specializing in "
    "local governance. Follow the user's instructions exactly; preserve JSON validity."
)

# Plain instructions layered on top of the policy markdown file.
instr = (
    "You can hear the meeting in the attached audio and read any PDFs or images.\n"
    "Use the Orbis section below to line up organization ids when it helps; use null if you are not sure.\n"
    "Stick to what is actually in the audio and documents—do not invent quotes.\n"
)
if orbis_blob:
    ref_block = "\n".join(["", "=== ORBIS_REFERENCE (keys = org_id) ===", orbis_blob])
else:
    ref_block = "\n".join(
        [
            "",
            "=== ORBIS_REFERENCE ===",
            "(no orbis_lookup_by_org_id.json in orbis_files/)",
        ]
    )

user = f"{POLICY_PROMPT}\n\n---\n{instr}{ref_block}"

raw = call_google_genai_multimodal(
    api_key=GEMINI_API_KEY,
    model=GENAI_MODEL,
    system_instruction=SYSTEM,
    user_text=user,
    media=media_paths,
    temperature=0.1,
    max_output_tokens=8192,
)
parsed = parse_policy_analysis_response(raw)
parsed["chunk_index"] = 0
parsed["chunk_total"] = 1
results = [parsed]

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
stem = f"analysis_{stamp}_multimodal"
(OUT_GEMMA_JSON / f"{stem}.raw.txt").write_text(parsed.get("raw") or "", encoding="utf-8")
if parsed.get("json_analysis") is not None:
    (OUT_GEMMA_JSON / f"{stem}.json").write_text(
        json.dumps(parsed["json_analysis"], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
if parsed.get("summary"):
    (OUT_SUMMARIES / f"{stem}.summary.md").write_text(parsed["summary"], encoding="utf-8")
if parsed.get("extra"):
    (OUT_GEMMA_JSON / f"{stem}.extra.md").write_text(parsed["extra"], encoding="utf-8")

print("Wrote:", OUT_GEMMA_JSON / f"{stem}.json")
if parsed.get("parse_error"):
    print("Parse warning:", parsed["parse_error"])
print("Done. JSON:", OUT_GEMMA_JSON, "| summaries:", OUT_SUMMARIES)

## 6) Optional: merge extra Orbis fields into the JSON

If you have the same lookup file on disk, this step copies matching rows onto each `organization` entry by `org_id` and writes a second JSON file per chunk.

In [ ]:
ORBIS_PATH = PIPE.orbis_files / "orbis_lookup_by_org_id.json"

if ORBIS_PATH.is_file():
    orbis = json.loads(ORBIS_PATH.read_text(encoding="utf-8"))
    for out in results:
        ja = out.get("json_analysis")
        if isinstance(ja, dict) and "_error" not in ja:
            merge_orbis_into_organizations(ja, orbis)
    for i, out in enumerate(results):
        ja = out.get("json_analysis")
        if isinstance(ja, dict) and "_error" not in ja:
            p = OUT_GEMMA_JSON / f"analysis_orbis_enriched_part_{i + 1:02d}.json"
            p.write_text(json.dumps(ja, indent=2, ensure_ascii=False), encoding="utf-8")
            print("Wrote", p)
    print("Merged Orbis profiles where keys matched org_id")
else:
    print("No orbis file at", ORBIS_PATH, "— skip")

## 7) If something goes wrong

- **Model name errors:** open AI Studio, copy a Gemma 4 id that your account can use, set `GOVERNANCE_GENAI_MODEL` to that string.
- **Payload too big:** use shorter meetings, fewer PDFs, or ask the provider about uploading very large files outside this notebook.
- **Messy JSON in the answer:** open the `.raw.txt` file in `02_gemma_json/` and compare it to what `policy_analysis_v1.md` asks for.

In [ ]:
# End of notebook — add your own cells below if you want charts, checks, or a second model pass.